<a href="https://colab.research.google.com/github/anandita-3217/EAI-Assignments/blob/main/EAI-Assignments%20/SimplexAgent/SE26MAID034_SimplexAgent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Develop a simplex agent to calculate the AQI for a given area using real life sensor information.

Input: Sensor data for various pollutants

Output:
1. AQI - the number
2. Class of AQI (Good, Moderate, Unhealthy for sensitive groups, Unhealthy, Very Unhealthy, Hazardous)

References:

https://www.airnow.gov/aqi/aqi-basics/ for classification of AQI


*Dataset* for Hyderabad from Kaggle  https://www.kaggle.com/datasets/nitirajkulkarni/hyderabad-in-1269843


In [1]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d nitirajkulkarni/hyderabad-in-1269843
!unzip -o hyderabad-in-1269843.zip -d aqi_data

cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
Dataset URL: https://www.kaggle.com/datasets/nitirajkulkarni/hyderabad-in-1269843
License(s): CC0-1.0
100% 53.9k/53.9k [00:00<00:00, 5.41MB/s]

Archive:  hyderabad-in-1269843.zip
  inflating: aqi_data/README.md      
  inflating: aqi_data/air_quality_historical.csv  
  inflating: aqi_data/city_info.csv  
  inflating: aqi_data/data_dictionary.csv  


In [2]:
import pandas as pd
df = pd.read_csv("aqi_data/air_quality_historical.csv")
print(f"shape of original dataframe: {df.shape}")
print(f"columns of original dataframe: {df.columns}")

shape of original dataframe: (1298, 12)
columns of original dataframe: Index(['date', 'pm10', 'pm2_5', 'carbon_monoxide', 'nitrogen_dioxide',
       'sulphur_dioxide', 'ozone', 'aerosol_optical_depth', 'dust', 'uv_index',
       'us_aqi', 'european_aqi'],
      dtype='object')


Dropping columns of calculated aqi

In [3]:
aqi_df = df.drop(columns=['us_aqi', 'european_aqi'])
print(f"New dataframe with required columns: {aqi_df.shape}")

New dataframe with required columns: (1298, 10)


Dropping all na or nan values

In [4]:
aqi_df.isna().sum()

,0
date,0
pm10,3
pm2_5,3
carbon_monoxide,3
nitrogen_dioxide,3
sulphur_dioxide,3
ozone,3
aerosol_optical_depth,3
dust,3
uv_index,3


In [5]:
aqi_df = aqi_df.dropna()
print(f"Shape of new data frame after removing nan or na values: {aqi_df.shape}")

Shape of new data frame after removing nan or na values: (1295, 10)


Standardize values so to calculate aqi

In [6]:
def ugm3_to_ppb(conc_ugm3, molecular_weight):
  """
    This function converts a pollutant concentration from µg/m3 to ppb,
    using the ideal gas molar volume constant at standard conditions (25°C, 1 atm)

    Parameters:
    conc_ugm3(float): concentration of the pollutant in µg/m3
    molecular_weight(float): molecular weight of the pollutant (g/mol),
      e.g. NO2 = 46.0055, O3 = 48.00

    Returns:
    conc_ppb(float): concentration of the pollutant in ppb
  """
  return (conc_ugm3 * 24.45 )/molecular_weight

In [7]:
def calculate_sub_index(conc, breakpoints):
  """
    This function accepts a pollutant concentration and its breakpoint table,
    and returns the corresponding AQI sub-index using linear interpolation

    Parameters:
    conc(float): concentration of a single pollutant
    breakpoints(list of tuples): list of (bp_lo, bp_hi, i_lo, i_hi) tuples,
      where bp_lo/bp_hi are the concentration bounds and i_lo/i_hi are the
      corresponding AQI bounds for that category

    Returns:
    sub_index(float): the interpolated AQI value for this pollutant,
      or None if conc doesn't fall within any breakpoint range
  """

  for bp_lo, bp_hi, i_lo, i_hi in breakpoints:
    if bp_lo <= conc <= bp_hi:
      return ((i_hi - i_lo)/(bp_hi-bp_lo)) * (conc-bp_lo) + i_lo
  return None

Breakpoints of the pattern:  (BP_lo, BP_hi, I_lo, I_hi)

- BP_lo:	Lower bound of the pollutant concentration for this category
- BP_hi:	Upper bound of the pollutant concentration for this category
- I_lo:	The AQI value that corresponds to BP_lo
- I_hi	The AQI value that corresponds to BP_hi

In [8]:
pm25_bp = [(0.0, 9.0, 0, 50), # Good
           (9.1, 35.4, 51, 100), # Moderate
            (35.5, 55.4, 101, 150), # Unhealthy for Sensitive Groups
           (55.5, 125.4, 151, 200), # Unhealthy
            (125.5, 225.4, 201, 300), # Very Unhealthy
             (225.5, 500.4, 301, 500) # Hazardous
             ]

pm10_bp = [(0, 54, 0, 50), (55, 154, 51, 100), (155, 254, 101, 150),
           (255, 354, 151, 200), (355, 424, 201, 300), (425, 604, 301, 500)]

no2_bp = [(0, 53, 0, 50), (54, 100, 51, 100), (101, 360, 101, 150),
          (361, 649, 151, 200), (650, 1249, 201, 300), (1250, 2049, 301, 500)]

o3_bp = [(0, 54, 0, 50), (55, 70, 51, 100), (71, 85, 101, 150),
         (86, 105, 151, 200), (106, 200, 201, 300)]


In [9]:
def calculate_aqi(pm10,pm2_5,no2,o3):
  """
    This fucntion accepts values of various pollutants and returns the value of aqi

    Parameters:
      pm10(float) : PM10 concentration in µg/m3 (24-hour average)
      pm2_5(float) : PM2.5 concentration in µg/m3 (24-hour average)
      no2(float) : NO2 concentration in µg/m3 (converted internally to ppb, 1-hour average)
      o3(float) : O3 concentration in µg/m3 (converted internally to ppb, 8-hour average)

    Returns:
      aqi(int): value of aqi from the given pollutants
  """
  no2_ppb = ugm3_to_ppb(no2, 46.0055)
  o3_ppb = ugm3_to_ppb(o3, 48.00)

  sub_indices = {
      "PM10": calculate_sub_index(pm10, pm10_bp),
      "PM2.5": calculate_sub_index(pm2_5, pm25_bp),
      "NO2": calculate_sub_index(no2_ppb, no2_bp),
      "O3": calculate_sub_index(o3_ppb, o3_bp),
  }

  valid = [v for v in sub_indices.values() if v is not None]
  if not valid:
    return None

  return round(max(valid))

In [10]:
def aqi_class(aqi):
  """
    This function accepts the value of aqi as a parameter and outputs a class of the aqi
    parameters:
      aqi(int): value of aqi as calculated from parameters

    returns:
      aqi_class(str): class of aqi depending on the value of aqi
    Good, Moderate, Unhealthy for sensitive groups, Unhealthy, Very Unhealthy, Hazardous

  """
  if 0 <= aqi <= 50:
    return "Good"
  elif 51 <= aqi <= 100:
    return "Moderate"
  elif 101 <= aqi <= 150:
    return "Unhealthy for sensitive groups"
  elif 151 <= aqi <= 200:
    return "Unhealthy"
  elif 201 <= aqi <= 300:
    return "Very Unhealthy"
  else:
    return "Hazardous"

In [11]:
aqi = calculate_aqi(pm10=60, pm2_5=40, no2=30, o3=50)
print(aqi)
print(aqi_class(aqi))

112
Unhealthy for sensitive groups
